In [144]:
import os
from pathlib import Path

from fraise.paths import data_root
from fraise.inference.artifacts import list_model_runs, load_run
from fraise.inference.scoring import score_text, top_next_tokens
from fraise.inference.generation import generate
from fraise.inference.benchmark import benchmark_generation

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
models_directory = data_root(os.environ.get("FRAISE_DATA_DIR", project_root / "data")) / "models"
runs = list_model_runs(models_directory)
for number, path in enumerate(runs, 0):
    print(number, path)


0 /home/munch/Projects/fraise/data/models/bbb
1 /home/munch/Projects/fraise/data/models/fraise_model
2 /home/munch/Projects/fraise/data/models/minsorhours
3 /home/munch/Projects/fraise/data/models/overfit
4 /home/munch/Projects/fraise/data/models/overfitfine
5 /home/munch/Projects/fraise/data/models/overfitthousand
6 /home/munch/Projects/fraise/data/models/overfittwo
7 /home/munch/Projects/fraise/data/models/stdtiedweights
8 /home/munch/Projects/fraise/data/models/testingthiscommit
9 /home/munch/Projects/fraise/data/models/unwantedchild
10 /home/munch/Projects/fraise/data/models/whyis


In [145]:
RUN_DIRECTORY = runs[5]  # Select the run you want to inspect.
CHECKPOINT = None  # Optional checkpoint filename; otherwise load the final model or latest step.
run = load_run(RUN_DIRECTORY, checkpoint=CHECKPOINT)
model, tokenizer = run.model, run.tokenizer
model.compile()
print(run.path, run.config)
print("Parameters:", sum(parameter.numel() for parameter in model.parameters()))


/home/munch/Projects/fraise/data/models/overfitthousand/overfitthousand.pt ModelConfig(vocab_size=8192, residual_stream_dim=512, head_dim=64, qk_dim=64, num_heads=12, swiglu_up_dim=1024, num_layers=12, max_seq_len=512, tie_word_embeddings=True)
Parameters: 42006528


# Inspect a saved Fraise run

Install notebook tools with `uv sync --extra notebooks`. Select a run directory below. The shared loader preserves its archived configuration, tokenizer, and weight tying. Set `FRAISE_DATA_DIR` when launching outside the project.


In [170]:
TEST_TEXT = "We cannot"
print(score_text(model, tokenizer, TEST_TEXT))
for token_id, probability, text in top_next_tokens(model, tokenizer, "The future of small language models is"):
    print(token_id, probability, repr(text))


{'tokens_scored': 3, 'truncated': False, 'loss': 9.778156280517578, 'perplexity': 17644.09208944156}
424 0.40756723284721375 ' not'
290 0.26565760374069214 ' to'
4583 0.0777696892619133 ' certainly'
2998 0.04177888110280037 ' trying'
2389 0.025236252695322037 ' getting'
327 0.02361065335571766 ' that'
260 0.021265633404254913 ' a'
5468 0.010821369476616383 ' concerned'
971 0.010812085121870041 ' being'
824 0.006569836288690567 ' very'


In [173]:
PROMPT = """Hmm,"""
text, ids = generate(model, tokenizer, PROMPT, max_new_tokens=100, temperature=0.0)
print(text)


Hmm, John Major says to speak to the recommendations of sports grounds, does the National Allas, etcetera by saying the Samaritans will be hand about again?
Well, like I'm going to take it out, I'm going to ask everybody who is not going to have too high, he should be given Oxford, to bring food; the better levels of women, and I think the health authority is organisading himself, so I'm going to be very aware that


## Full-context decoding throughput

This measures greedy generation without a KV cache. Warmup runs are excluded; fixed-length timed runs continue through EOS.


In [148]:
print(benchmark_generation(model, tokenizer, PROMPT, max_new_tokens=32, warmup_runs=1, timed_runs=3))


{'prompt_tokens': 53, 'tokens_per_run': 32, 'timed_runs': 3, 'total_seconds': 0.16937798900471535, 'tokens_per_second': 566.7796657883772}
